# logsumexp-cross-entropy — ex2: closed-form CE gradient: (softmax - onehot) / B

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `logsumexp-cross-entropy`. Running the final beacon cell reports progress against the `Loss: logsumexp cross-entropy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: logsumexp cross-entropy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`logsumexp-cross-entropy`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "logsumexp-cross-entropy"
DD_SUBTOPIC = "Loss: logsumexp cross-entropy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-entropy gradient identity — deepening

The gradient of cross-entropy w.r.t. the logits has a closed form that makes the backward pass trivial — no autograd machinery needed:

```
dL/d(logits[i, k]) = (softmax(logits[i])[k] - onehot(target[i])[k]) / B
```

for the mean-reduced CE loss. Equivalently in tensor form:

```python
probs = softmax(logits, dim=-1)
onehot = F.one_hot(target, num_classes=C).float()
grad_logits = (probs - onehot) / B
```

**Why this is the killer identity.** The naive chain rule routes through logsumexp and the per-sample picker — both differentiable, but you'd have a 3-step backward graph. The closed form collapses to a single op, which is what `nn.CrossEntropyLoss.backward` actually does internally.

### Exercise 2 — closed-form CE gradient: (softmax - onehot) / B

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the closed-form identity `dL/dlogits = (softmax(logits) - onehot(target)) / B` to compute the cross-entropy gradient in one step, then validate against torch.autograd as the witness.
> Keywords: cross-entropy, gradient, softmax, onehot, closed-form
> ```

**KCs targeted:** `logsumexp-cross-entropy`, `softmax-minus-onehot-grad`

Implement `cross_entropy_grad(logits, target)`. The gradient of mean-reduced cross-entropy w.r.t. the logits, in CLOSED FORM:

```
grad[i, k] = (softmax(logits[i])[k] - 1{k == target[i]}) / B
```

Inputs:
- `logits`: shape `(B, C)`, float.
- `target`: shape `(B,)`, integer class indices in `[0, C)`.

Output: shape `(B, C)`, float — same shape as `logits`.

Recipe:
1. `probs = t.softmax(logits, dim=-1)` — shape `(B, C)`.
2. `onehot = F.one_hot(target, num_classes=logits.shape[1]).float()` — shape `(B, C)`.
3. `grad = (probs - onehot) / B`.
4. Return `grad`.

**Do NOT call `.backward()`** on a torch loss to compute this — the drill is the closed form. The test cell cross-checks against autograd as a witness (which you call from the TEST, not from your implementation).

In [ ]:
def cross_entropy_grad(logits, target):
    import torch.nn.functional as F
    B, C = logits.shape
    probs = t.softmax(logits, dim=-1)               # (B, C)
    onehot = F.one_hot(target, num_classes=C).to(probs.dtype)  # (B, C)
    return (probs - onehot) / B


<details><summary>Solution</summary>

```python
def cross_entropy_grad(logits, target):
    import torch.nn.functional as F
    B, C = logits.shape
    probs = t.softmax(logits, dim=-1)               # (B, C)
    onehot = F.one_hot(target, num_classes=C).to(probs.dtype)  # (B, C)
    return (probs - onehot) / B
```

**Why the closed form exists.** Cross-entropy + softmax is the canonical 'exponential-family + log-link' pair from generalized linear models. For any such pair the gradient of the negative log-likelihood w.r.t. the natural parameter (here, logits) is `predicted - observed` — i.e. `softmax(logits) - onehot(target)`. The `/B` comes purely from mean-reduction.

**Sign convention.** `grad[i, target[i]] < 0` says SGD will push the target logit UP (because `p - lr * grad` adds magnitude). Conversely `grad[i, k] > 0` for non-target classes pushes those logits DOWN. Net effect: each step nudges the probabilities toward the one-hot target.

**Numerical advantage.** Computing this directly bypasses the logsumexp + arange-fancy-index path entirely. Frameworks (`nn.CrossEntropyLoss`) fuse logsumexp-forward with softmax-minus-onehot-backward — never building the full graph.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()